In [0]:
from pyspark.sql.functions import *

In [0]:
df_silver = spark.table("uk_ecommerce.silver.online_retail")

In [0]:
display(df_silver)

## Check for cancellations

- Verify whether cancelled invoices, after removing the C prefix, have a corresponding original invoice number in the non-cancelled orders.

In [0]:
# Create cancellation orders DF
cancellations_orders = df_silver.filter(df_silver["InvoiceNo"].startswith("C"))
cancellations_orders = cancellations_orders.withColumn("InvoiceNoWithoutC", regexp_replace("InvoiceNo", "^C", "")).distinct()

# Create DF without cancellations
orders_without_cancellations = df_silver.filter(~df_silver["InvoiceNo"].startswith("C")).distinct()

# Check whether the invoice numbers from cancellations_orders, after removing the C prefix, exist in orders_without_cancellations
joined_tables = cancellations_orders.join(orders_without_cancellations, cancellations_orders["InvoiceNoWithoutC"] == orders_without_cancellations["InvoiceNo"], "left").select(cancellations_orders["InvoiceNoWithoutC"], orders_without_cancellations["InvoiceNo"]).distinct()

display(joined_tables)

A total of 3,654 cancellation invoices were identified. For none of them was a non-cancelled invoice found with the same number after removing the C prefix. Therefore, the dataset does not allow us to conclude that an original sale is retained as a second invoice with the same number, nor does it allow us to link a cancellation to its original sale based solely on the InvoiceNo.

- Find a case where one of the items in an order was cancelled, and print all the lines from that invoice, including both the cancelled and non-cancelled items.

In [0]:
cancelled_items = df_silver.filter(df_silver["InvoiceNo"].startswith("C"))
non_cancelled_items = df_silver.filter(~df_silver["InvoiceNo"].startswith("C"))

In [0]:
c = (
    cancelled_items
    .withColumn("cancellation_line_id", monotonically_increasing_id())
    .alias("c")
)

s = non_cancelled_items.alias("s")

candidate_matches = (
    c.join(
        s,
        (
            (c["CustomerID"] == s["CustomerID"]) &
            (c["StockCode"] == s["StockCode"]) &
            (c["InvoiceDate"] > s["InvoiceDate"]) &
            (abs(c["Quantity"]) <= s["Quantity"])
        ),
        "inner"
    )
    .select(
        c["cancellation_line_id"],

        c["InvoiceNo"].alias("cancellation_invoice_no"),
        c["CustomerID"].alias("customer_id"),
        c["StockCode"].alias("stock_code"),
        c["Description"].alias("cancelled_description"),
        c["Quantity"].alias("cancelled_quantity"),
        c["UnitPrice"].alias("cancelled_unit_price"),
        c["InvoiceDate"].alias("cancellation_date"),

        s["InvoiceNo"].alias("original_invoice_no"),
        s["Description"].alias("original_description"),
        s["Quantity"].alias("original_quantity"),
        s["UnitPrice"].alias("original_unit_price"),
        s["InvoiceDate"].alias("original_sale_date")
    )
    .withColumn(
        "days_between_sale_and_cancellation",
        datediff(
            col("cancellation_date"),
            col("original_sale_date")
        )
    )
)

In [0]:
display(candidate_matches)

In [0]:
from pyspark.sql.window import Window

window_closest_sale = Window.partitionBy(
    "cancellation_line_id"
).orderBy(
    col("days_between_sale_and_cancellation").asc(),
    col("original_sale_date").desc()
)

closest_sale_candidates = (
    candidate_matches
    .withColumn(
        "candidate_rank",
        row_number().over(window_closest_sale)
    )
    .filter(col("candidate_rank") == 1)
)

display(
    closest_sale_candidates
    .orderBy("days_between_sale_and_cancellation")
)

In [0]:
original_invoice = non_cancelled_items.filter(
    col("InvoiceNo") == "542724"
)

cancellation_invoice = cancelled_items.filter(
    col("InvoiceNo") == "C542913"
)

display(original_invoice.orderBy("StockCode"))
display(cancellation_invoice.orderBy("StockCode"))

A possible partial cancellation was identified. The original invoice contained 13 different products. Only two products appeared in the cancellation invoice: StockCode 21041, with 6 units purchased and 1 unit cancelled, and StockCode 22197, with 36 units purchased and 2 units cancelled. The remaining products from the original invoice did not appear in the cancellation record.

## Analysis by Country

- What the Country with the most customers?

In [0]:
country_with_most_customer = (
    df_silver
    .groupBy("Country")
    .agg(count_distinct("CustomerID").alias("TotalCustomers"))
    .orderBy(desc("TotalCustomers"))
    .limit(1)
)

display(country_with_most_customer)

The United Kingdom has by far the highest number of distinct customers, with 3,950 unique CustomerIDs — consistent with this being a UK-based online retailer and the dataset's primary market.

- Is the Country field always formatted the same way? (all uppercase or lowercase? no typos?)

In [0]:
# Check the occurrence for each field to identify typos
check_country_names = (
    df_silver
    .withColumn("country_normalized", lower(trim(col("Country"))))
    .groupBy("country_normalized")
    .agg(
        count("*").alias("occurrences"),
        count_distinct("Country").alias("number_of_variations"),
        collect_set("Country").alias("original_variations")
    )
    .filter(col("number_of_variations") > 1)
    .orderBy(desc("number_of_variations"), desc("occurrences"))
)

display(check_country_names)


No formatting inconsistencies were found in the Country field: after normalizing case and trimming whitespace, every distinct country name maps to exactly one original spelling. This confirms the field is well standardized, with no typos or casing variations to correct.

## Product Descriptions

- Find 2 codes that have different descriptions (same code with different descriptions)

In [0]:
# StockCode that have multiples descriptions
products_with_multiple_descriptions = (
    df_silver.groupBy("StockCode")
    .agg(count_distinct("Description").alias("Occurrences"),
         collect_set("Description").alias("Descriptions")
    )
    .filter("Occurrences > 1")
)

display(products_with_multiple_descriptions)

A total of 213 StockCode values have more than one associated description. Most cases (as in rows 1-15 above) involve minor formatting differences, such as extra whitespace, alternate word order, abbreviations, or singular/plural variants (e.g. "GLITTER CHRISTMAS HEART " vs "GLITTER HEART DECORATION"). A smaller subset shows more substantial wording changes (e.g. StockCode 22602: "CHRISTMAS RETROSPOT HEART WOOD" vs "RETROSPOT WOODEN HEART DECORATION"), which required manual review to confirm they refer to the same product before assigning a single canonical description.

- Normalize the description column containing two or more descriptions so that it contains only a single description.

In [0]:
df_normalized_description = spark.read.format("csv").load("/Volumes/uk_ecommerce/bronze/reference/normalized_descriptions.csv", header=True, inferSchema=True)

display(df_normalized_description)

In [0]:
# Create a new updated df_silver with Description column normalized
df_silver_updated = (
    df_silver
    .join(df_normalized_description, on="StockCode", how="left")
    .withColumn("Description",
                coalesce(col("NormalizedDescriptions"), col("Description")))
    .drop("NormalizedDescriptions")
)

display(df_silver_updated)

## Metrics Calculation

### Analysis between purchases and cancellations

In [0]:
cancelled_purchases = df_silver_updated.filter(df_silver_updated.InvoiceNo.startswith("C"))
non_cancelled_purchases = df_silver_updated.filter(~df_silver_updated.InvoiceNo.startswith("C"))

In [0]:
c = (
    cancelled_purchases
    .withColumn("cancellation_line_id", monotonically_increasing_id())
    .alias("c")
)

s = non_cancelled_purchases.alias("s")

candidate_matches = (
    c.join(
        s,
        (
            (c["CustomerID"] == s["CustomerID"]) &
            (c["StockCode"] == s["StockCode"]) &
            (c["InvoiceDate"] > s["InvoiceDate"]) &
            (abs(c["Quantity"]) <= s["Quantity"])
        ),
        "inner"
    )
    .select(
        c["cancellation_line_id"],

        c["InvoiceNo"].alias("cancellation_invoice_no"),
        c["CustomerID"].alias("customer_id"),
        c["StockCode"].alias("stock_code"),
        c["Description"].alias("cancelled_description"),
        c["Quantity"].alias("cancelled_quantity"),
        c["UnitPrice"].alias("cancelled_unit_price"),
        c["InvoiceDate"].alias("cancellation_date"),

        s["InvoiceNo"].alias("original_invoice_no"),
        s["Description"].alias("original_description"),
        s["Quantity"].alias("original_quantity"),
        s["UnitPrice"].alias("original_unit_price"),
        s["InvoiceDate"].alias("original_sale_date")
    )
    .withColumn(
        "days_between_sale_and_cancellation",
        datediff(
            col("cancellation_date"),
            col("original_sale_date")
        )
    )
)

In [0]:
window_closest_sale = Window.partitionBy(
    "cancellation_line_id"
).orderBy(
    col("days_between_sale_and_cancellation").asc(),
    col("original_sale_date").desc()
)

closest_sale_candidates = (
    candidate_matches
    .withColumn(
        "candidate_rank",
        row_number().over(window_closest_sale)
    )
    .filter(col("candidate_rank") == 1)
)

display(closest_sale_candidates)

- Average days between a purchase and cancellation

In [0]:
days_between_purchase_cancellation = closest_sale_candidates.select(round(avg("days_between_sale_and_cancellation"), 2).alias("avg_days_between_purchase_and_cancellation"),
                                                                            round(percentile_approx("days_between_sale_and_cancellation", 0.5), 2).alias("median_days_between_purchase_and_cancellation"),
                                                                            count("*").alias("quantity"))



display(days_between_purchase_cancellation)


Across 7,069 matched cancellation-sale pairs, the average time between a purchase and its cancellation is 29.27 days, while the median is only 12 days. The gap between mean and median indicates a right-skewed distribution: most cancellations happen relatively soon after the original purchase, but a smaller number of cases with much longer gaps pull the average upward.

- Max and min — days between purchase and cancellation
    - Max: X days
    - Min: Y days

In [0]:
max_min_days_between_purchase_cancellation = closest_sale_candidates.select(max("days_between_sale_and_cancellation"), min("days_between_sale_and_cancellation"))

display(max_min_days_between_purchase_cancellation)


- Max and min — days between purchase and cancellation
    - Max: 368 days
    - Min: 1 day

The 368-day maximum is close to a full year, meaning at least one cancellation was matched to a sale nearly a year prior — worth checking against the dataset's own date range (see check below) to confirm this isn't an artifact of a mismatched pair rather than a genuine year-long gap. The 1-day minimum shows some cancellations happen almost immediately after purchase.

In [0]:
check_data_min_max = df_silver_updated.agg(min("InvoiceDate"), max("InvoiceDate"))

display(check_data_min_max)

The dataset spans from 2010-12-01 to 2011-12-09, a range of roughly 373 days. The 368-day maximum found in the previous cell falls within this range, confirming it reflects a genuine long-standing purchase that was eventually cancelled near the end of the observed period, rather than a matching error.

- Histogram — distribution of days between purchase and cancellation

In [0]:
import matplotlib.pyplot as plt

In [0]:
distribution_days_between_purchase_cancellation = closest_sale_candidates.select("days_between_sale_and_cancellation").toPandas()

plt.hist(distribution_days_between_purchase_cancellation["days_between_sale_and_cancellation"], bins=100)
plt.title("Days Between Sale and Cancellation")
plt.xlabel("Days Between Sale and Cancellation")
plt.ylabel("Frequency (Number of Cancellations)")
plt.show()

The distribution of days between sale and cancellation is heavily right-skewed: the vast majority of cancellations occur within the first 0-20 days after purchase, with a sharp peak near day 0. A long tail extends out to nearly a year, but with very low frequency — confirming that the median (12 days) is a much more representative figure than the average (29.27 days) for describing typical cancellation behavior.

In [0]:
q1, q3 = closest_sale_candidates.approxQuantile("days_between_sale_and_cancellation", [0.25, 0.75], 0.01)
iqr = q3 - q1
outlier = q3 + 1.5 * iqr

print(f"Quartil 1 (Q1): {q1}")
print(f"Quartil 3 (Q3): {q3}")
print(f"Intervalo Interquartil (IQR): {iqr}")
print(f"Outlier: {outlier}")

Using the IQR method (Q1 = 6 days, Q3 = 29 days, IQR = 23), the outlier threshold for cancellation delay is 63.5 days. Any cancellation matched to a sale more than 63.5 days earlier is considered a statistical outlier — a useful, data-driven cutoff for flagging unusually long gaps rather than relying on an arbitrary visual threshold from the histogram.

In [0]:
distribution_days_between_purchase_cancellation_outliers = closest_sale_candidates.filter(
    col("days_between_sale_and_cancellation") > outlier).toPandas()

plt.hist(distribution_days_between_purchase_cancellation_outliers["days_between_sale_and_cancellation"], bins=30)
plt.title("Days Between Sale and Cancellation (Statistical Outliers, IQR > 63.5 Days)")
plt.xlabel("Days Between Sale and Cancellation")
plt.ylabel("Frequency (Number of Cancellations)")
plt.show()

Zooming into the outlier tail (>63.5 days) reveals it isn't a flat noise floor: it's still concentrated in the 60-100 day range and decays gradually from there, with a small resurgence of cases near 350 days (close to the dataset's own time span). This suggests the "outliers" are a mix of genuinely slow cancellations rather than random noise, and the small cluster near 350 days likely overlaps with the 368-day maximum identified earlier.

In [0]:
total_outliers = len(distribuition_days_between_purchase_cancellation_outliers)
total_geral = closest_sale_candidates.count()

print(f"Total de outliers: {total_outliers}")
print(f"Total geral: {total_geral}")
print(f"Percentual de outliers: {(total_outliers / total_geral) * 100:.2f}%")

Outliers (cancellations more than 63.5 days after the original sale) represent 834 out of 7,069 matched cancellations — 11.80% of the total. While this is a meaningful minority, the vast majority of cancellations (88.2%) still occur within the expected, short-delay window, reinforcing that late cancellations are a real but secondary pattern rather than the norm.